In [11]:
!pip install statsmodels

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB 6.3 MB/s eta 0:00:02
   -- ------------------------------------- 0.5/9.5 MB 5.2 MB/s eta 0:00:02
   ---- ----------------------------------- 1.1/9.5 MB 7.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.7/9.5 MB 9.1 MB/s eta 0:00:01
   --------- ------------------------------ 2.2/9.5 MB 8.8 MB/s eta 0:00:01
   ----------- ---------------------------- 2.7/9.5 MB 9.1 MB/s eta 0:00:01
   ------------- -------------------------- 3.1/9.5 MB 9.0 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.5 MB 8.4 MB/s eta 0:00:01
   --------------- ------------------------ 3.7/9.5 MB 7.9 MB/s eta 0:00:01
   ----------------- ---------------------- 4.1/9.5 MB 7.9 MB/s eta 0:00:01
   ------------------- -------------------- 4.6/9.5 MB 7.9 MB/s eta 0:00:01
   -------------------- ------------------- 4.9/9.5 MB 7.8 MB/s eta 0:00:01
   ----------------

In [14]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ★ 경로 설정 (이 줄만 확인하세요) ★
DATA_DIR = r"C:\Users\LG\DScover\26-1 가이드프로젝트"

def p(filename):
    return os.path.join(DATA_DIR, filename)

def p_meta(filename):
    return os.path.join(DATA_DIR, "meta", filename)

# 1. 데이터 로드
train           = pd.read_csv(p("train.csv"),                   encoding="utf-8-sig")
wholesale_train = pd.read_csv(p("TRAIN_전국도매_2018-2021.csv"), encoding="utf-8-sig")
submission      = pd.read_csv(p("sample_submission.csv"),       encoding="utf-8-sig")
print(f"train: {train.shape}, wholesale: {wholesale_train.shape}, submission: {submission.shape}")

# 2. 타겟 품목 정의
target_items = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    {'품목명': '사과',        '품종명': '후지',        '거래단위': '10 개',     '등급': '상품'},
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자','등급': '상'},
    {'품목명': '배',          '품종명': '신고',        '거래단위': '10 개',     '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',     '등급': '상품'},
    {'품목명': '무',          '품종명': '무',          '거래단위': '20키로상자','등급': '상'},
    {'품목명': '상추',        '품종명': '청',          '거래단위': '100 g',     '등급': '상품'},
    {'품목명': '배추',        '품종명': '배추',        '거래단위': '10키로망대','등급': '상'},
    {'품목명': '양파',        '품종명': '양파',        '거래단위': '1키로',     '등급': '상'},
    {'품목명': '대파',        '품종명': '대파(일반)',  '거래단위': '1키로단',   '등급': '상'},
]
wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '고추'}
lag_dict = {
    '건고추': None, '사과': 4, '감자': 2, '배': 3,
    '깐마늘(국산)': 3, '무': 4, '상추': 2, '배추': 2, '양파': 2, '대파': 2,
}

# 3. 칼만 필터 결측치 대치
print("\n[Step 1] 칼만 필터 결측치 대치 시작...")
all_time_df = pd.DataFrame({'시점': sorted(train['시점'].unique())})
imputed_results = {}

for item in target_items:
    name = f"{item['품목명']}_{item['품종명']}"
    cond_price = ((train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) &
                  (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급']))
    df_price = train[cond_price][['시점', '평균가격(원)']]

    search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
    df_supply = (wholesale_train[wholesale_train['품목명'] == search_name]
                 .groupby('시점')['총반입량(kg)'].sum().reset_index())

    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')
    df_merged = pd.merge(df_merged, df_supply, on='시점', how='left')
    df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"  [{name}] 데이터 0건. 건너뜁니다.")
        continue
    try:
        model = sm.tsa.UnobservedComponents(
            endog=df_merged['평균가격(원)'], level='local linear trend',
            seasonal=36, autoregressive=1, exog=df_merged['총반입량(kg)'])
        res = model.fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        imputed_results[name] = df_merged
        print(f"  [{name}] 완료! ({valid_count}건 → 144건)")
    except Exception as e:
        print(f"  [{name}] 에러: {e}")

print("✅ 칼만 필터 결측치 대치 완료!\n")

# 4. TEST 추론
print("[Step 2] TEST 추론 시작...")
test_ids = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

for test_id in test_ids:
    print(f"  [{test_id}]...", end=" ")
    test_num = test_id.split('_')[1]
    try:
        test           = pd.read_csv(p(f"{test_id}.csv"),                    encoding="utf-8-sig")
        wholesale_test = pd.read_csv(p_meta(f"TEST_전국도매_{test_num}.csv"), encoding="utf-8-sig")
    except FileNotFoundError as e:
        print(f"파일 없음 → 건너뜁니다."); continue

    for item in target_items:
        col_name = item['품목명']
        lag_val  = lag_dict[col_name]

        cond_tr = ((train['품목명'] == item['품목명']) & (train['품종명'] == item['품종명']) &
                   (train['거래단위'] == item['거래단위']) & (train['등급'] == item['등급']))
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = ((test['품목명'] == item['품목명']) & (test['품종명'] == item['품종명']) &
                   (test['거래단위'] == item['거래단위']) & (test['등급'] == item['등급']))
        df_te_p = test[cond_te][['시점', '평균가격(원)']]
        df_p    = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        df_tr_s = (wholesale_train[wholesale_train['품목명'] == search_name]
                   .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_te_s = (wholesale_test[wholesale_test['품목명'] == search_name]
                   .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)

        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].fillna(0)

        if lag_val is None:
            exog_train = exog_future = None
        else:
            recent_mean = df_merged['총반입량(kg)'].tail(3).mean()
            extended    = pd.concat([df_merged['총반입량(kg)'], pd.Series([recent_mean]*3)], ignore_index=True)
            shifted     = extended.shift(lag_val).bfill()
            exog_train  = shifted.iloc[:len(df_merged)].values
            exog_future = shifted.iloc[len(df_merged):].values

        try:
            params = {'endog': df_merged['평균가격(원)'], 'level': 'local linear trend',
                      'seasonal': 36, 'autoregressive': 1}
            if exog_train is not None: params['exog'] = exog_train
            res = sm.tsa.UnobservedComponents(**params).fit(disp=False)
            fc  = np.maximum(res.forecast(steps=3, exog=exog_future).values if exog_future is not None
                             else res.forecast(steps=3).values, 0)
            result_sub.loc[f'{test_id}+1순', col_name] = fc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = fc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = fc[2]
        except:
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0
    print("완료")

# 5. 저장
result_sub.reset_index(inplace=True)
out_path = p("submission_SSM.csv")
result_sub.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"\n저장 완료 → {out_path}")
print(f"0인 값 개수: {(result_sub.iloc[:,1:] == 0).sum().sum()}")
print(result_sub.head(6).to_string())

train: (29376, 7), wholesale: (176014, 22), submission: (75, 11)

[Step 1] 칼만 필터 결측치 대치 시작...
  [건고추_화건] 완료! (144건 → 144건)
  [사과_후지] 완료! (125건 → 144건)
  [감자_감자 수미] 완료! (144건 → 144건)
  [배_신고] 완료! (144건 → 144건)
  [깐마늘(국산)_깐마늘(국산)] 완료! (144건 → 144건)
  [무_무] 완료! (144건 → 144건)
  [상추_청] 완료! (144건 → 144건)
  [배추_배추] 완료! (144건 → 144건)
  [양파_양파] 완료! (144건 → 144건)
  [대파_대파(일반)] 완료! (144건 → 144건)
✅ 칼만 필터 결측치 대치 완료!

[Step 2] TEST 추론 시작...
  [TEST_00]... 완료
  [TEST_01]... 완료
  [TEST_02]... 완료
  [TEST_03]... 완료
  [TEST_04]... 완료
  [TEST_05]... 완료
  [TEST_06]... 완료
  [TEST_07]... 완료
  [TEST_08]... 완료
  [TEST_09]... 완료
  [TEST_10]... 완료
  [TEST_11]... 완료
  [TEST_12]... 완료
  [TEST_13]... 완료
  [TEST_14]... 완료
  [TEST_15]... 완료
  [TEST_16]... 완료
  [TEST_17]... 완료
  [TEST_18]... 완료
  [TEST_19]... 완료
  [TEST_20]... 완료
  [TEST_21]... 완료
  [TEST_22]... 완료
  [TEST_23]... 완료
  [TEST_24]... 완료

저장 완료 → C:\Users\LG\DScover\26-1 가이드프로젝트\submission_SSM.csv
0인 값 개수: 8
           시점            감자            건고추     

# 반입량과 가격을 분리해서 처리

가격(endog)과 반입량(exog)은 성격이 달라서 처리 방법도 달라야 해요.

반입량(exog) → 선형 보간 ✅

칼만 필터는 exog에 NaN이 있으면 에러가 나요. 그래서 반드시 채워야 하는데, 지금 0으로 채우는 게 문제예요. 선형 보간이 가장 현실적이에요.

가격(endog) → NaN 그대로 + 칼만이 추론 ✅

가격은 칼만 필터가 알아서 확률적으로 추론해줘요. 억지로 채울 필요가 없어요.

## -> 아예 차이가 없음

In [16]:
# ============================================================
# 농산물 가격 예측 — 상태 공간 모델 (SSM + 칼만 필터)
# 팀원 코드를 Jupyter 환경에 맞게 변환
# ============================================================

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────
# 0. 경로 설정 (★ 여기만 수정하세요 ★)
# ────────────────────────────────────────────────
DATA_DIR = r"C:\Users\LG\DScover\26-1 가이드프로젝트"

def p(filename):
    """파일 경로 생성 헬퍼"""
    return os.path.join(DATA_DIR, filename)

def p_meta(filename):
    """meta 폴더 경로 생성 헬퍼"""
    return os.path.join(DATA_DIR, "meta", filename)


# ────────────────────────────────────────────────
# 1. 공통 데이터 로드
# ────────────────────────────────────────────────
train           = pd.read_csv(p("train.csv"),                    encoding="utf-8-sig")
wholesale_train = pd.read_csv(p("TRAIN_전국도매_2018-2021.csv"),  encoding="utf-8-sig")
submission      = pd.read_csv(p("sample_submission.csv"),        encoding="utf-8-sig")

print(f"train:           {train.shape}")
print(f"wholesale_train: {wholesale_train.shape}")
print(f"submission:      {submission.shape}")


# ────────────────────────────────────────────────
# 2. 타겟 10개 품목 정의
# ────────────────────────────────────────────────
target_items = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    {'품목명': '사과',        '품종명': '후지',        '거래단위': '10 개',      '등급': '상품'},
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배',          '품종명': '신고',        '거래단위': '10 개',      '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',      '등급': '상품'},
    {'품목명': '무',          '품종명': '무',          '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추',        '품종명': '청',          '거래단위': '100 g',      '등급': '상품'},
    {'품목명': '배추',        '품종명': '배추',        '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파',        '품종명': '양파',        '거래단위': '1키로',      '등급': '상'},
    {'품목명': '대파',        '품종명': '대파(일반)',  '거래단위': '1키로단',    '등급': '상'},
]

# 도매시장 품목명 매핑 (깐마늘/건고추는 전국도매에서 다른 이름 사용)
wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추':       '고추',
}

# 품목별 분석된 반입량 시차(Lag)
lag_dict = {
    '건고추':       None,  # 외생변수 제외
    '사과':         4,
    '감자':         2,
    '배':           3,
    '깐마늘(국산)': 3,
    '무':           4,
    '상추':         2,
    '배추':         2,
    '양파':         2,
    '대파':         2,
}


# ────────────────────────────────────────────────
# 3. 칼만 필터 결측치 대치 (train 데이터 전처리)
# ────────────────────────────────────────────────
print("\n[Step 1] 칼만 필터 결측치 대치 시작...")

all_time_steps = train['시점'].unique()
all_time_df    = pd.DataFrame({'시점': all_time_steps}).sort_values('시점').reset_index(drop=True)

imputed_results = {}

for item in target_items:
    name = f"{item['품목명']}_{item['품종명']}"

    # 가격 데이터 추출
    cond_price = (
        (train['품목명']    == item['품목명']) &
        (train['품종명']    == item['품종명']) &
        (train['거래단위']  == item['거래단위']) &
        (train['등급']      == item['등급'])
    )
    df_price = train[cond_price][['시점', '평균가격(원)']]

    # 전국도매 총반입량 추출 및 집계
    search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
    cond_supply = (wholesale_train['품목명'] == search_name)
    df_supply   = (wholesale_train[cond_supply]
                   .groupby('시점')['총반입량(kg)'].sum()
                   .reset_index())

    # 전체 시점 뼈대에 병합
    df_merged = pd.merge(all_time_df, df_price,  on='시점', how='left')
    df_merged = pd.merge(df_merged,   df_supply,  on='시점', how='left')
    df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"  [{name}] 데이터 0건. 건너뜁니다.")
        continue

    try:
        model = sm.tsa.UnobservedComponents(
            endog  = df_merged['평균가격(원)'],
            level  = 'local linear trend',
            seasonal     = 36,
            autoregressive = 1,
            exog   = df_merged['총반입량(kg)'],
        )
        res             = model.fit(disp=False)
        smoothed_prices = res.predict()

        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(smoothed_prices)
        imputed_results[name]     = df_merged

        print(f"  [{name}] 칼만 대치 완료! (원본 {valid_count}건 → 144건 복원)")

    except Exception as e:
        print(f"  [{name}] 에러: {e}")

print("✅ 칼만 필터 결측치 대치 완료!\n")


# ────────────────────────────────────────────────
# 4. 대치 결과 저장 (선택)
# ────────────────────────────────────────────────
final_dfs = []
for name, df in imputed_results.items():
    parts = name.split('_', 1)
    item_name = parts[0]
    item_kind = parts[1] if len(parts) > 1 else ''
    df_clean = df.copy()
    df_clean.insert(0, '품목명', item_name)
    df_clean.insert(1, '품종명', item_kind)
    # 총반입량 컬럼이 없을 경우 대비
    cols = [c for c in ['시점', '품목명', '품종명', '총반입량(kg)', '평균가격(원)', '대치가격(원)']
            if c in df_clean.columns]
    final_dfs.append(df_clean[cols])

if final_dfs:
    final_imputed_data = pd.concat(final_dfs, ignore_index=True)
    out = p("train_imputed_kalman.csv")
    final_imputed_data.to_csv(out, index=False, encoding='utf-8-sig')
    print(f"대치 데이터 저장 완료 → {out}")
    print(final_imputed_data.head())


# ────────────────────────────────────────────────
# 5. TEST 추론 → submission 생성
# ────────────────────────────────────────────────
print("\n[Step 2] TEST 추론 시작...")

test_ids   = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

print(f"총 {len(test_ids)}개 테스트 데이터셋 예측 시작...\n")

for test_id in test_ids:
    print(f"  [{test_id}] 예측 중...", end=" ")
    test_num = test_id.split('_')[1]   # '00', '01' 등

    # 파일 로드 (meta 폴더 안에 있는 전국도매 test 파일)
    try:
        test           = pd.read_csv(p(f"{test_id}.csv"),              encoding="utf-8-sig")
        wholesale_test = pd.read_csv(p_meta(f"TEST_전국도매_{test_num}.csv"), encoding="utf-8-sig")
    except FileNotFoundError as e:
        print(f"파일 없음 → 건너뜁니다. ({e})")
        continue

    for item in target_items:
        col_name = item['품목명']
        lag_val  = lag_dict[col_name]

        # 가격 데이터 추출 (train + test 이어붙이기)
        cond_tr = (
            (train['품목명']   == item['품목명']) &
            (train['품종명']   == item['품종명']) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = (
            (test['품목명']   == item['품목명']) &
            (test['품종명']   == item['품종명']) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        df_te_p = test[cond_te][['시점', '평균가격(원)']]
        df_p    = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        # 반입량 데이터 추출 (train + test)
        search_name = wholesale_name_map.get(item['품목명'], item['품목명'])
        df_tr_s = (wholesale_train[wholesale_train['품목명'] == search_name]
                   .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_te_s = (wholesale_test[wholesale_test['품목명'] == search_name]
                   .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_s = pd.concat([df_tr_s, df_te_s], ignore_index=True)

        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()

        # 품목별 시차(Lag) 적용
        if lag_val is None:
            exog_train  = None
            exog_future = None
        else:
            recent_mean     = df_merged['총반입량(kg)'].tail(3).mean()
            extended_supply = pd.concat(
                [df_merged['총반입량(kg)'], pd.Series([recent_mean] * 3)],
                ignore_index=True
            )
            shifted_supply  = extended_supply.shift(lag_val).bfill()
            exog_train      = shifted_supply.iloc[:len(df_merged)].values
            exog_future     = shifted_supply.iloc[len(df_merged):].values

        # 상태 공간 모델 학습 및 예측
        try:
            model_params = {
                'endog':           df_merged['평균가격(원)'],
                'level':           'local linear trend',
                'seasonal':        36,
                'autoregressive':  1,
            }
            if exog_train is not None:
                model_params['exog'] = exog_train

            model     = sm.tsa.UnobservedComponents(**model_params)
            res       = model.fit(disp=False)

            if exog_future is not None:
                forecasts = res.forecast(steps=3, exog=exog_future)
            else:
                forecasts = res.forecast(steps=3)

            forecasts = np.maximum(forecasts.values, 0)  # 음수 방지

            result_sub.loc[f'{test_id}+1순', col_name] = forecasts[0]
            result_sub.loc[f'{test_id}+2순', col_name] = forecasts[1]
            result_sub.loc[f'{test_id}+3순', col_name] = forecasts[2]

        except Exception as e:
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0

    print("완료")

# ────────────────────────────────────────────────
# 6. 저장
# ────────────────────────────────────────────────
result_sub.reset_index(inplace=True)
out_path = p("submission_SSM.csv_exog_endog.csv")
result_sub.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n저장 완료 → {out_path}")
print(f"0인 값 개수: {(result_sub.iloc[:, 1:] == 0).sum().sum()}")
print(result_sub.head(6).to_string())

train:           (29376, 7)
wholesale_train: (176014, 22)
submission:      (75, 11)

[Step 1] 칼만 필터 결측치 대치 시작...
  [건고추_화건] 에러: exog contains inf or nans
  [사과_후지] 칼만 대치 완료! (원본 125건 → 144건 복원)
  [감자_감자 수미] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [배_신고] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [깐마늘(국산)_깐마늘(국산)] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [무_무] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [상추_청] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [배추_배추] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [양파_양파] 칼만 대치 완료! (원본 144건 → 144건 복원)
  [대파_대파(일반)] 칼만 대치 완료! (원본 144건 → 144건 복원)
✅ 칼만 필터 결측치 대치 완료!

대치 데이터 저장 완료 → C:\Users\LG\DScover\26-1 가이드프로젝트\train_imputed_kalman.csv
         시점 품목명 품종명    총반입량(kg)  평균가격(원)  대치가격(원)
0  201801상순  사과  후지  16493192.6  20361.0  20361.0
1  201801중순  사과  후지  22596097.8  20359.0  20359.0
2  201801하순  사과  후지  23485500.0  20653.0  20653.0
3  201802상순  사과  후지  51539576.4  20563.0  20563.0
4  201802중순  사과  후지   7038032.0  21779.0  21779.0

[Step 2] TEST 추론 시작...
총 25개 테스트 데이터셋 예측 시작...

  [TEST_00] 예측 중... 완료
  [TEST_

# EDA 기반 개선 1

수정 1 — EDA 기반 exog 전략 재조정
품목상관계수변경이유배-0.17exog 제거태풍·명절 수요가 더 결정적무-0.11exog 제거기상 충격이 주요 변수배추+0.10exog 제거양의 상관(반직관적), 김장 수요 결정양파-0.07exog 제거거의 무상관, 저장·수입량 영향이 더 큼
수정 2 — 사과 홍로/후지 분리 모델링

기존: 후지만 단일 모델
변경: 홍로(9~10월 조생종) + 후지(연중 저장 출하) 각각 SSM → 평균

In [17]:
# ============================================================
# 농산물 가격 예측 — 상태 공간 모델 (SSM + 칼만 필터) v3
# 수정 내역: EDA 기반 exog 전략 재조정 + 사과 홍로/후지 분리 모델링
# ============================================================

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────
# 0. 경로 설정 (★ 여기만 수정하세요 ★)
# ────────────────────────────────────────────────
DATA_DIR = r"C:\Users\LG\DScover\26-1 가이드프로젝트"

def p(filename):
    return os.path.join(DATA_DIR, filename)

def p_meta(filename):
    return os.path.join(DATA_DIR, "meta", filename)


# ────────────────────────────────────────────────
# 1. 공통 데이터 로드
# ────────────────────────────────────────────────
train           = pd.read_csv(p("train.csv"),                   encoding="utf-8-sig")
wholesale_train = pd.read_csv(p("TRAIN_전국도매_2018-2021.csv"), encoding="utf-8-sig")
submission      = pd.read_csv(p("sample_submission.csv"),       encoding="utf-8-sig")

print(f"train: {train.shape} | wholesale: {wholesale_train.shape} | submission: {submission.shape}")


# ────────────────────────────────────────────────
# 2. 타겟 품목 정의
# ────────────────────────────────────────────────
target_items = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    # [수정] 사과는 별도 처리 (홍로/후지 분리) → 아래 apple_items 참고
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배',          '품종명': '신고',        '거래단위': '10 개',      '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',      '등급': '상품'},
    {'품목명': '무',          '품종명': '무',          '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추',        '품종명': '청',          '거래단위': '100 g',      '등급': '상품'},
    {'품목명': '배추',        '품종명': '배추',        '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파',        '품종명': '양파',        '거래단위': '1키로',      '등급': '상'},
    {'품목명': '대파',        '품종명': '대파(일반)',  '거래단위': '1키로단',    '등급': '상'},
]

# [수정] 사과 홍로/후지 분리 모델링용 별도 리스트
apple_items = [
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '사과', '품종명': '홍로', '거래단위': '10 개', '등급': '상품'},
]

wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추':       '고추',
}

# ────────────────────────────────────────────────
# [수정 1] EDA 기반 품목별 exog 사용 여부 결정
# 반입량-가격 상관관계가 약한 품목은 exog 제거 (노이즈 방지)
# ────────────────────────────────────────────────
# corr 기준:
#   상추 -0.43 ✅ / 감자 -0.29 ✅ / 사과(후지) -0.29 ✅ / 깐마늘 -0.21 ✅ / 대파 -0.20 ✅
#   배   -0.17 ❌ / 무  -0.11 ❌ / 배추 +0.10 ❌ / 양파 -0.07 ❌ / 건고추 None (기존 유지)
USE_EXOG = {
    '건고추':       False,   # 기존: 외생변수 없음 유지
    '사과':         True,    # 후지 -0.29 (보통)
    '감자':         True,    # -0.29 (보통)
    '배':           False,   # [수정] -0.17 약함 → 제거
    '깐마늘(국산)': True,    # -0.21 (보통)
    '무':           False,   # [수정] -0.11 매우 약함 → 제거
    '상추':         True,    # -0.43 (강함)
    '배추':         False,   # [수정] +0.10 양의 상관 → 제거
    '양파':         False,   # [수정] -0.07 거의 없음 → 제거
    '대파':         True,    # -0.20 (보통)
}

# 품목별 반입량 시차(Lag) — exog 사용 품목만 의미 있음
lag_dict = {
    '건고추':       None,
    '사과':         4,
    '감자':         2,
    '배':           None,   # [수정] exog 제거로 None
    '깐마늘(국산)': 3,
    '무':           None,   # [수정] exog 제거로 None
    '상추':         2,
    '배추':         None,   # [수정] exog 제거로 None
    '양파':         None,   # [수정] exog 제거로 None
    '대파':         2,
}


# ────────────────────────────────────────────────
# 3. 공통 함수: exog 준비 (선형 보간 + 시차 적용)
# ────────────────────────────────────────────────
def prepare_exog(df_merged, col_name, lag_val, steps=3):
    """
    반입량 선형 보간 후 시차 적용.
    exog_train: 학습용, exog_future: 예측용(steps개)
    """
    supply = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()
    recent_mean = supply.tail(3).mean()
    extended    = pd.concat([supply, pd.Series([recent_mean] * steps)], ignore_index=True)
    shifted     = extended.shift(lag_val).bfill()
    exog_train  = shifted.iloc[:len(df_merged)].values
    exog_future = shifted.iloc[len(df_merged):].values
    return exog_train, exog_future


# ────────────────────────────────────────────────
# 4. 칼만 필터 결측치 대치 (train 전처리)
# ────────────────────────────────────────────────
print("\n[Step 1] 칼만 필터 결측치 대치 시작...")

all_time_df = pd.DataFrame({'시점': sorted(train['시점'].unique())})
imputed_results = {}

# 일반 품목 (사과 제외)
for item in target_items:
    name      = f"{item['품목명']}_{item['품종명']}"
    col_name  = item['품목명']
    use_exog  = USE_EXOG.get(col_name, False)

    cond_price = (
        (train['품목명']   == item['품목명']) &
        (train['품종명']   == item['품종명']) &
        (train['거래단위'] == item['거래단위']) &
        (train['등급']     == item['등급'])
    )
    df_price = train[cond_price][['시점', '평균가격(원)']]
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')

    # exog 사용 품목만 반입량 추가
    if use_exog:
        search_name = wholesale_name_map.get(col_name, col_name)
        df_supply   = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_merged   = pd.merge(df_merged, df_supply, on='시점', how='left')
        # [수정] 선형 보간 (기존 fillna(0) 대신)
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"  [{name}] 데이터 0건. 건너뜁니다.")
        continue

    try:
        model_params = {
            'endog':           df_merged['평균가격(원)'],
            'level':           'local linear trend',
            'seasonal':        36,
            'autoregressive':  1,
        }
        if use_exog:
            model_params['exog'] = df_merged['총반입량(kg)']

        res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        imputed_results[name]     = df_merged
        exog_label = "(exog O)" if use_exog else "(exog X)"
        print(f"  [{name}] 완료 {exog_label} ({valid_count}건 → 144건)")

    except Exception as e:
        print(f"  [{name}] 에러: {e}")


# ────────────────────────────────────────────────
# [수정 2] 사과 홍로/후지 분리 칼만 대치
# ────────────────────────────────────────────────
print("\n  [사과] 홍로/후지 분리 칼만 대치...")
apple_imputed = {}

for item in apple_items:
    name = f"사과_{item['품종명']}"
    cond = (
        (train['품목명']   == '사과') &
        (train['품종명']   == item['품종명']) &
        (train['거래단위'] == item['거래단위']) &
        (train['등급']     == item['등급'])
    )
    df_price  = train[cond][['시점', '평균가격(원)']]
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')

    # 사과 반입량 (품종별로 따로 집계)
    df_supply = (wholesale_train[
                    (wholesale_train['품목명'] == '사과') &
                    (wholesale_train['품종명'] == item['품종명'])
                 ].groupby('시점')['총반입량(kg)'].sum().reset_index())
    df_merged = pd.merge(df_merged, df_supply, on='시점', how='left')
    df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"    [{name}] 데이터 0건. 건너뜁니다.")
        continue

    try:
        res = sm.tsa.UnobservedComponents(
            endog          = df_merged['평균가격(원)'],
            level          = 'local linear trend',
            seasonal       = 36,
            autoregressive = 1,
            exog           = df_merged['총반입량(kg)'],
        ).fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        apple_imputed[item['품종명']] = df_merged
        print(f"    [{name}] 완료! ({valid_count}건 → 144건)")
    except Exception as e:
        print(f"    [{name}] 에러: {e}")

print("✅ 칼만 필터 결측치 대치 완료!\n")


# ────────────────────────────────────────────────
# 5. TEST 추론 → submission 생성
# ────────────────────────────────────────────────
print("[Step 2] TEST 추론 시작...")

test_ids   = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

for test_id in test_ids:
    print(f"  [{test_id}]...", end=" ")
    test_num = test_id.split('_')[1]

    try:
        test           = pd.read_csv(p(f"{test_id}.csv"),                    encoding="utf-8-sig")
        wholesale_test = pd.read_csv(p_meta(f"TEST_전국도매_{test_num}.csv"), encoding="utf-8-sig")
    except FileNotFoundError:
        print("파일 없음 → 건너뜁니다.")
        continue

    # ── 일반 품목 예측 (사과 제외) ──────────────────
    for item in target_items:
        col_name = item['품목명']
        lag_val  = lag_dict[col_name]
        use_exog = USE_EXOG.get(col_name, False)

        cond_tr = (
            (train['품목명']   == item['품목명']) &
            (train['품종명']   == item['품종명']) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = (
            (test['품목명']   == item['품목명']) &
            (test['품종명']   == item['품종명']) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        df_te_p  = test[cond_te][['시점', '평균가격(원)']]
        df_p     = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_merged = df_p.copy()

        exog_train = exog_future = None
        if use_exog and lag_val is not None:
            search_name = wholesale_name_map.get(col_name, col_name)
            df_tr_s = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
            df_te_s = (wholesale_test[wholesale_test['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
            df_s      = pd.concat([df_tr_s, df_te_s], ignore_index=True)
            df_merged = pd.merge(df_p, df_s, on='시점', how='left')
            exog_train, exog_future = prepare_exog(df_merged, col_name, lag_val)

        try:
            model_params = {
                'endog':          df_merged['평균가격(원)'],
                'level':          'local linear trend',
                'seasonal':       36,
                'autoregressive': 1,
            }
            if exog_train is not None:
                model_params['exog'] = exog_train

            res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
            fc  = np.maximum(
                (res.forecast(steps=3, exog=exog_future).values
                 if exog_future is not None
                 else res.forecast(steps=3).values),
                0
            )
            result_sub.loc[f'{test_id}+1순', col_name] = fc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = fc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = fc[2]

        except Exception:
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0

    # ── [수정 2] 사과 홍로/후지 분리 예측 후 평균 ──
    apple_forecasts = {}
    for item in apple_items:
        variety = item['품종명']
        cond_tr = (
            (train['품목명']   == '사과') &
            (train['품종명']   == variety) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = (
            (test['품목명']   == '사과') &
            (test['품종명']   == variety) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        df_te_p  = test[cond_te][['시점', '평균가격(원)']]
        df_p     = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        # 품종별 반입량
        df_tr_s = (wholesale_train[
                        (wholesale_train['품목명'] == '사과') &
                        (wholesale_train['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_te_s = (wholesale_test[
                        (wholesale_test['품목명'] == '사과') &
                        (wholesale_test['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_s      = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')
        df_merged['총반입량(kg)'] = df_merged['총반입량(kg)'].interpolate(method='linear').ffill().bfill()

        try:
            exog_train, exog_future = prepare_exog(df_merged, '사과', lag_dict['사과'])
            res = sm.tsa.UnobservedComponents(
                endog          = df_merged['평균가격(원)'],
                level          = 'local linear trend',
                seasonal       = 36,
                autoregressive = 1,
                exog           = exog_train,
            ).fit(disp=False)
            fc = np.maximum(res.forecast(steps=3, exog=exog_future).values, 0)
            apple_forecasts[variety] = fc
        except Exception:
            apple_forecasts[variety] = np.array([0.0, 0.0, 0.0])

    # 홍로/후지 예측값 평균 → 사과 최종 예측
    if apple_forecasts:
        fc_apple = np.mean(list(apple_forecasts.values()), axis=0)
    else:
        fc_apple = np.array([0.0, 0.0, 0.0])

    result_sub.loc[f'{test_id}+1순', '사과'] = fc_apple[0]
    result_sub.loc[f'{test_id}+2순', '사과'] = fc_apple[1]
    result_sub.loc[f'{test_id}+3순', '사과'] = fc_apple[2]

    print("완료")


# ────────────────────────────────────────────────
# 6. 저장
# ────────────────────────────────────────────────
result_sub.reset_index(inplace=True)
# [수정] 파일명에 수정 내용 반영
out_path = p("submission_SSM_exog_revised_apple_split.csv")
result_sub.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n저장 완료 → {out_path}")
print(f"0인 값 개수: {(result_sub.iloc[:,1:] == 0).sum().sum()}")
print(result_sub.head(6).to_string())


# ============================================================
# ## 수정 내역 (v2 → v3)
#
# ### 수정 1: EDA 기반 품목별 exog(반입량) 전략 재조정
# - 반입량-가격 상관관계 분석 결과를 반영해
#   상관관계가 약한 품목은 exog를 제거함
#
# | 품목     | 상관계수 | 이전 | 변경 후 | 이유 |
# |----------|---------|------|---------|------|
# | 배       | -0.17   | exog 사용 | **제거** | 상관 약함, 태풍·명절 수요가 더 결정적 |
# | 무       | -0.11   | exog 사용 | **제거** | 기상 충격이 주요 변수, 반입량 설명력 낮음 |
# | 배추     | +0.10   | exog 사용 | **제거** | 양의 상관(반직관적), 김장 수요가 결정적 |
# | 양파     | -0.07   | exog 사용 | **제거** | 거의 무상관, 저장 출하·수입량 영향이 더 큼 |
# | 상추     | -0.43   | exog 사용 | 유지    | 강한 음의 상관 |
# | 감자     | -0.29   | exog 사용 | 유지    | 보통 음의 상관 |
# | 깐마늘   | -0.21   | exog 사용 | 유지    | 보통 음의 상관 |
# | 대파     | -0.20   | exog 사용 | 유지    | 보통 음의 상관 | 
#
# ### 수정 2: 사과 홍로/후지 분리 모델링
# - 기존: 사과를 후지 한 품종으로만 예측
# - 변경: 홍로(9~10월 조생종)와 후지(연중 저장 출하)를
#         각각 별도 SSM 모델로 학습 후 예측값 평균
# - 이유: EDA에서 두 품종의 출하 패턴이 완전히 달라
#         하나의 모델로 묶으면 계절성 추정이 왜곡됨
#
# ### 수정 3: 파일명 영어로 변경
# - submission_SSM_exog_revised_apple_split.csv
# ============================================================

train: (29376, 7) | wholesale: (176014, 22) | submission: (75, 11)

[Step 1] 칼만 필터 결측치 대치 시작...
  [건고추_화건] 완료 (exog X) (144건 → 144건)
  [감자_감자 수미] 완료 (exog O) (144건 → 144건)
  [배_신고] 완료 (exog X) (144건 → 144건)
  [깐마늘(국산)_깐마늘(국산)] 완료 (exog O) (144건 → 144건)
  [무_무] 완료 (exog X) (144건 → 144건)
  [상추_청] 완료 (exog O) (144건 → 144건)
  [배추_배추] 완료 (exog X) (144건 → 144건)
  [양파_양파] 완료 (exog X) (144건 → 144건)
  [대파_대파(일반)] 완료 (exog O) (144건 → 144건)

  [사과] 홍로/후지 분리 칼만 대치...
    [사과_후지] 완료! (125건 → 144건)
    [사과_홍로] 완료! (19건 → 144건)
✅ 칼만 필터 결측치 대치 완료!

[Step 2] TEST 추론 시작...
  [TEST_00]... 완료
  [TEST_01]... 완료
  [TEST_02]... 완료
  [TEST_03]... 완료
  [TEST_04]... 완료
  [TEST_05]... 완료
  [TEST_06]... 완료
  [TEST_07]... 완료
  [TEST_08]... 완료
  [TEST_09]... 완료
  [TEST_10]... 완료
  [TEST_11]... 완료
  [TEST_12]... 완료
  [TEST_13]... 완료
  [TEST_14]... 완료
  [TEST_15]... 완료
  [TEST_16]... 완료
  [TEST_17]... 완료
  [TEST_18]... 완료
  [TEST_19]... 완료
  [TEST_20]... 완료
  [TEST_21]... 완료
  [TEST_22]... 완료
  [TEST_23]... 완료
  [TES

# 시도 2 [fail]
-> 
기존: 과거 가격 흐름만 보고 예측

시도2: 평년 평균가격 추가 입력


즉

exog(외생변수)에 반입량만 넣던 걸 -> 반입량 + 평년가격 두 개 동시에 넣음

특히 반입량 상관관계가 약해 직전에서 exog를 제거했던 배,무,배추,양파도 평년가격 추가

In [1]:
# ============================================================
# 농산물 가격 예측 — 상태 공간 모델 (SSM + 칼만 필터) v4
# v3 대비 수정: 평년 평균가격을 모든 품목에 exog로 추가
# ============================================================

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────
# 0. 경로 설정 (★ 여기만 수정하세요 ★)
# ────────────────────────────────────────────────
DATA_DIR = r"C:\Users\LG\DScover\26-1 가이드프로젝트"

def p(filename):
    return os.path.join(DATA_DIR, filename)

def p_meta(filename):
    return os.path.join(DATA_DIR, "meta", filename)


# ────────────────────────────────────────────────
# 1. 공통 데이터 로드
# ────────────────────────────────────────────────
train           = pd.read_csv(p("train.csv"),                   encoding="utf-8-sig")
wholesale_train = pd.read_csv(p("TRAIN_전국도매_2018-2021.csv"), encoding="utf-8-sig")
submission      = pd.read_csv(p("sample_submission.csv"),       encoding="utf-8-sig")

print(f"train: {train.shape} | wholesale: {wholesale_train.shape} | submission: {submission.shape}")


# ────────────────────────────────────────────────
# 2. 타겟 품목 정의
# ────────────────────────────────────────────────
target_items = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배',          '품종명': '신고',        '거래단위': '10 개',      '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',      '등급': '상품'},
    {'품목명': '무',          '품종명': '무',          '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추',        '품종명': '청',          '거래단위': '100 g',      '등급': '상품'},
    {'품목명': '배추',        '품종명': '배추',        '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파',        '품종명': '양파',        '거래단위': '1키로',      '등급': '상'},
    {'품목명': '대파',        '품종명': '대파(일반)',  '거래단위': '1키로단',    '등급': '상'},
]

apple_items = [
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '사과', '품종명': '홍로', '거래단위': '10 개', '등급': '상품'},
]

wholesale_name_map = {
    '깐마늘(국산)': '마늘',
    '건고추':       '고추',
}

# EDA 기반 반입량 exog 사용 여부 (v3와 동일)
USE_SUPPLY_EXOG = {
    '건고추':       False,
    '사과':         True,
    '감자':         True,
    '배':           False,
    '깐마늘(국산)': True,
    '무':           False,
    '상추':         True,
    '배추':         False,
    '양파':         False,
    '대파':         True,
}

lag_dict = {
    '건고추':       None,
    '사과':         4,
    '감자':         2,
    '배':           None,
    '깐마늘(국산)': 3,
    '무':           None,
    '상추':         2,
    '배추':         None,
    '양파':         None,
    '대파':         2,
}


# ────────────────────────────────────────────────
# 3. 공통 함수
# ────────────────────────────────────────────────

def interp_fill(series):
    """선형 보간 + 양끝 ffill/bfill"""
    return series.interpolate(method='linear').ffill().bfill()


def prepare_supply_exog(df_merged, lag_val, steps=3):
    """반입량 선형 보간 + 시차 적용 → (exog_train, exog_future)"""
    supply      = interp_fill(df_merged['총반입량(kg)'])
    recent_mean = supply.tail(3).mean()
    extended    = pd.concat([supply, pd.Series([recent_mean] * steps)], ignore_index=True)
    shifted     = extended.shift(lag_val).bfill()
    return shifted.iloc[:len(df_merged)].values, shifted.iloc[len(df_merged):].values


# [수정 v4] 평년가격 exog 준비 함수
def prepare_norm_price_exog(df_merged, norm_price_col='평년 평균가격(원)', steps=3):
    """
    평년 평균가격을 exog로 준비.
    - train: 시점별 평년가격 그대로 사용
    - future(예측 구간 3개): 마지막 3개 평균으로 연장
    결측치는 선형 보간으로 처리.
    """
    norm_price  = interp_fill(df_merged[norm_price_col].astype(float))
    recent_mean = norm_price.tail(3).mean()
    extended    = pd.concat([norm_price, pd.Series([recent_mean] * steps)], ignore_index=True)
    return extended.iloc[:len(df_merged)].values, extended.iloc[len(df_merged):].values


def build_exog(arrays_train, arrays_future):
    """
    여러 exog 배열을 2D matrix로 합치기.
    arrays_train: list of 1D np.array (학습용)
    arrays_future: list of 1D np.array (예측용)
    """
    if not arrays_train:
        return None, None
    X_train = np.column_stack(arrays_train)
    X_future = np.column_stack(arrays_future)
    return X_train, X_future


# ────────────────────────────────────────────────
# 4. 칼만 필터 결측치 대치 (train 전처리)
# ────────────────────────────────────────────────
print("\n[Step 1] 칼만 필터 결측치 대치 시작...")

all_time_df = pd.DataFrame({'시점': sorted(train['시점'].unique())})
imputed_results = {}

# 일반 품목 (사과 제외)
for item in target_items:
    name     = f"{item['품목명']}_{item['품종명']}"
    col_name = item['품목명']

    # 가격 + 평년가격 추출
    cond_price = (
        (train['품목명']   == item['품목명']) &
        (train['품종명']   == item['품종명']) &
        (train['거래단위'] == item['거래단위']) &
        (train['등급']     == item['등급'])
    )
    # [수정 v4] 평년 평균가격도 함께 추출
    price_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
    price_cols = [c for c in price_cols if c in train.columns]
    df_price  = train[cond_price][price_cols]
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')

    # 반입량 추가 (사용 품목만)
    if USE_SUPPLY_EXOG.get(col_name, False):
        search_name = wholesale_name_map.get(col_name, col_name)
        df_supply   = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_merged   = pd.merge(df_merged, df_supply, on='시점', how='left')
        df_merged['총반입량(kg)'] = interp_fill(df_merged['총반입량(kg)'])

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"  [{name}] 데이터 0건. 건너뜁니다.")
        continue

    # [수정 v4] exog 구성: 반입량(선택) + 평년가격(전 품목 필수)
    exog_arrays = []
    if USE_SUPPLY_EXOG.get(col_name, False) and '총반입량(kg)' in df_merged.columns:
        exog_arrays.append(interp_fill(df_merged['총반입량(kg)']).values)
    if '평년 평균가격(원)' in df_merged.columns:
        exog_arrays.append(interp_fill(df_merged['평년 평균가격(원)'].astype(float)).values)

    exog_matrix = np.column_stack(exog_arrays) if exog_arrays else None

    try:
        model_params = {
            'endog':           df_merged['평균가격(원)'],
            'level':           'local linear trend',
            'seasonal':        36,
            'autoregressive':  1,
        }
        if exog_matrix is not None:
            model_params['exog'] = exog_matrix

        res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        imputed_results[name]     = df_merged

        exog_label = f"(exog: 반입량{'O' if USE_SUPPLY_EXOG.get(col_name) else 'X'} + 평년가격O)"
        print(f"  [{name}] 완료 {exog_label} ({valid_count}건 → 144건)")

    except Exception as e:
        print(f"  [{name}] 에러: {e}")


# 사과 홍로/후지 분리 칼만 대치
print("\n  [사과] 홍로/후지 분리 칼만 대치...")
apple_imputed = {}

for item in apple_items:
    name    = f"사과_{item['품종명']}"
    variety = item['품종명']
    cond = (
        (train['품목명']   == '사과') &
        (train['품종명']   == variety) &
        (train['거래단위'] == item['거래단위']) &
        (train['등급']     == item['등급'])
    )
    price_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
    price_cols = [c for c in price_cols if c in train.columns]
    df_price  = train[cond][price_cols]
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')

    df_supply = (wholesale_train[
                    (wholesale_train['품목명'] == '사과') &
                    (wholesale_train['품종명'] == variety)
                 ].groupby('시점')['총반입량(kg)'].sum().reset_index())
    df_merged = pd.merge(df_merged, df_supply, on='시점', how='left')
    df_merged['총반입량(kg)'] = interp_fill(df_merged['총반입량(kg)'])

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        print(f"    [{name}] 데이터 0건. 건너뜁니다.")
        continue

    # [수정 v4] 반입량 + 평년가격 2개 exog
    exog_arrays = [interp_fill(df_merged['총반입량(kg)']).values]
    if '평년 평균가격(원)' in df_merged.columns:
        exog_arrays.append(interp_fill(df_merged['평년 평균가격(원)'].astype(float)).values)
    exog_matrix = np.column_stack(exog_arrays)

    try:
        res = sm.tsa.UnobservedComponents(
            endog          = df_merged['평균가격(원)'],
            level          = 'local linear trend',
            seasonal       = 36,
            autoregressive = 1,
            exog           = exog_matrix,
        ).fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        apple_imputed[variety]    = df_merged
        print(f"    [{name}] 완료! (반입량 + 평년가격 exog, {valid_count}건 → 144건)")
    except Exception as e:
        print(f"    [{name}] 에러: {e}")

print("✅ 칼만 필터 결측치 대치 완료!\n")


# ────────────────────────────────────────────────
# 5. TEST 추론 → submission 생성
# ────────────────────────────────────────────────
print("[Step 2] TEST 추론 시작...")

test_ids   = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

for test_id in test_ids:
    print(f"  [{test_id}]...", end=" ")
    test_num = test_id.split('_')[1]

    try:
        test           = pd.read_csv(p(f"{test_id}.csv"),                    encoding="utf-8-sig")
        wholesale_test = pd.read_csv(p_meta(f"TEST_전국도매_{test_num}.csv"), encoding="utf-8-sig")
    except FileNotFoundError:
        print("파일 없음 → 건너뜁니다.")
        continue

    # ── 일반 품목 예측 (사과 제외) ──────────────────
    for item in target_items:
        col_name = item['품목명']
        lag_val  = lag_dict[col_name]

        cond_tr = (
            (train['품목명']   == item['품목명']) &
            (train['품종명']   == item['품종명']) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        price_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
        price_cols = [c for c in price_cols if c in train.columns]
        df_tr_p = train[cond_tr][price_cols]

        cond_te = (
            (test['품목명']   == item['품목명']) &
            (test['품종명']   == item['품종명']) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        te_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
        te_cols = [c for c in te_cols if c in test.columns]
        df_te_p  = test[cond_te][te_cols]
        df_p     = pd.concat([df_tr_p, df_te_p], ignore_index=True)
        df_merged = df_p.copy()

        # 반입량 추가 (사용 품목만)
        if USE_SUPPLY_EXOG.get(col_name, False) and lag_val is not None:
            search_name = wholesale_name_map.get(col_name, col_name)
            df_tr_s = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
            df_te_s = (wholesale_test[wholesale_test['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
            df_s      = pd.concat([df_tr_s, df_te_s], ignore_index=True)
            df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        # [수정 v4] exog 구성: 반입량(시차적용, 선택) + 평년가격(전 품목 필수)
        train_arrays, future_arrays = [], []

        if USE_SUPPLY_EXOG.get(col_name, False) and lag_val is not None and '총반입량(kg)' in df_merged.columns:
            et, ef = prepare_supply_exog(df_merged, lag_val)
            train_arrays.append(et)
            future_arrays.append(ef)

        if '평년 평균가격(원)' in df_merged.columns:
            nt, nf = prepare_norm_price_exog(df_merged)
            train_arrays.append(nt)
            future_arrays.append(nf)

        exog_train, exog_future = build_exog(train_arrays, future_arrays)

        try:
            model_params = {
                'endog':          df_merged['평균가격(원)'],
                'level':          'local linear trend',
                'seasonal':       36,
                'autoregressive': 1,
            }
            if exog_train is not None:
                model_params['exog'] = exog_train

            res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
            fc  = np.maximum(
                res.forecast(steps=3, exog=exog_future).values
                if exog_future is not None
                else res.forecast(steps=3).values,
                0
            )
            result_sub.loc[f'{test_id}+1순', col_name] = fc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = fc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = fc[2]

        except Exception:
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0

    # ── 사과 홍로/후지 분리 예측 후 평균 ──
    apple_forecasts = {}
    for item in apple_items:
        variety = item['품종명']

        cond_tr = (
            (train['품목명']   == '사과') &
            (train['품종명']   == variety) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        price_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
        price_cols = [c for c in price_cols if c in train.columns]
        df_tr_p = train[cond_tr][price_cols]

        cond_te = (
            (test['품목명']   == '사과') &
            (test['품종명']   == variety) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        te_cols = ['시점', '평균가격(원)', '평년 평균가격(원)']
        te_cols = [c for c in te_cols if c in test.columns]
        df_te_p  = test[cond_te][te_cols]
        df_p     = pd.concat([df_tr_p, df_te_p], ignore_index=True)

        df_tr_s = (wholesale_train[
                        (wholesale_train['품목명'] == '사과') &
                        (wholesale_train['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_te_s = (wholesale_test[
                        (wholesale_test['품목명'] == '사과') &
                        (wholesale_test['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_s      = pd.concat([df_tr_s, df_te_s], ignore_index=True)
        df_merged = pd.merge(df_p, df_s, on='시점', how='left')

        train_arrays, future_arrays = [], []
        et, ef = prepare_supply_exog(df_merged, lag_dict['사과'])
        train_arrays.append(et); future_arrays.append(ef)

        if '평년 평균가격(원)' in df_merged.columns:
            nt, nf = prepare_norm_price_exog(df_merged)
            train_arrays.append(nt); future_arrays.append(nf)

        exog_train, exog_future = build_exog(train_arrays, future_arrays)

        try:
            res = sm.tsa.UnobservedComponents(
                endog          = df_merged['평균가격(원)'],
                level          = 'local linear trend',
                seasonal       = 36,
                autoregressive = 1,
                exog           = exog_train,
            ).fit(disp=False)
            fc = np.maximum(res.forecast(steps=3, exog=exog_future).values, 0)
            apple_forecasts[variety] = fc
        except Exception:
            apple_forecasts[variety] = np.array([0.0, 0.0, 0.0])

    fc_apple = np.mean(list(apple_forecasts.values()), axis=0) if apple_forecasts else np.zeros(3)
    result_sub.loc[f'{test_id}+1순', '사과'] = fc_apple[0]
    result_sub.loc[f'{test_id}+2순', '사과'] = fc_apple[1]
    result_sub.loc[f'{test_id}+3순', '사과'] = fc_apple[2]

    print("완료")


# ────────────────────────────────────────────────
# 6. 저장
# ────────────────────────────────────────────────
result_sub.reset_index(inplace=True)
out_path = p("submission_SSM_v4_norm_price_exog.csv")
result_sub.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n저장 완료 → {out_path}")
print(f"0인 값 개수: {(result_sub.iloc[:,1:] == 0).sum().sum()}")
print(result_sub.head(6).to_string())


# ============================================================
# ## 수정 내역 (v3 → v4)
#
# ### 수정: 평년 평균가격(원)을 모든 품목에 exog로 추가
#
# **왜 평년가격이 좋은 exog인가?**
# - train.csv와 test.csv 모두에 `평년 평균가격(원)` 컬럼이 존재
#   → 미래 데이터 누수(data leakage) 없이 안전하게 사용 가능
# - "이 시기에는 원래 이 가격대" 라는 계절 기준선을 모델에
#   직접 알려주는 역할 → SSM의 계절성 추정 보조
# - 반입량과 달리 상관관계가 약한 품목(배, 무, 배추, 양파)도
#   평년가격은 항상 의미 있는 참조값이 됨
#
# **exog 구성 변화 요약**
#
# | 품목         | v3 exog           | v4 exog                        |
# |--------------|-------------------|--------------------------------|
# | 건고추       | 없음              | **평년가격**                   |
# | 감자         | 반입량(lag=2)     | 반입량(lag=2) + **평년가격**   |
# | 배           | 없음              | **평년가격**                   |
# | 깐마늘       | 반입량(lag=3)     | 반입량(lag=3) + **평년가격**   |
# | 무           | 없음              | **평년가격**                   |
# | 상추         | 반입량(lag=2)     | 반입량(lag=2) + **평년가격**   |
# | 배추         | 없음              | **평년가격**                   |
# | 양파         | 없음              | **평년가격**                   |
# | 대파         | 반입량(lag=2)     | 반입량(lag=2) + **평년가격**   |
# | 사과(후지)   | 반입량(lag=4)     | 반입량(lag=4) + **평년가격**   |
# | 사과(홍로)   | 반입량(lag=4)     | 반입량(lag=4) + **평년가격**   |
#
# ### 파일명
# - submission_SSM_v4_norm_price_exog.csv
# ============================================================

train: (29376, 7) | wholesale: (176014, 22) | submission: (75, 11)

[Step 1] 칼만 필터 결측치 대치 시작...
  [건고추_화건] 완료 (exog: 반입량X + 평년가격O) (144건 → 144건)
  [감자_감자 수미] 완료 (exog: 반입량O + 평년가격O) (144건 → 144건)
  [배_신고] 완료 (exog: 반입량X + 평년가격O) (144건 → 144건)
  [깐마늘(국산)_깐마늘(국산)] 완료 (exog: 반입량O + 평년가격O) (144건 → 144건)
  [무_무] 완료 (exog: 반입량X + 평년가격O) (144건 → 144건)
  [상추_청] 완료 (exog: 반입량O + 평년가격O) (144건 → 144건)
  [배추_배추] 완료 (exog: 반입량X + 평년가격O) (144건 → 144건)
  [양파_양파] 완료 (exog: 반입량X + 평년가격O) (144건 → 144건)
  [대파_대파(일반)] 완료 (exog: 반입량O + 평년가격O) (144건 → 144건)

  [사과] 홍로/후지 분리 칼만 대치...
    [사과_후지] 완료! (반입량 + 평년가격 exog, 125건 → 144건)
    [사과_홍로] 완료! (반입량 + 평년가격 exog, 19건 → 144건)
✅ 칼만 필터 결측치 대치 완료!

[Step 2] TEST 추론 시작...
  [TEST_00]... 완료
  [TEST_01]... 완료
  [TEST_02]... 완료
  [TEST_03]... 완료
  [TEST_04]... 완료
  [TEST_05]... 완료
  [TEST_06]... 완료
  [TEST_07]... 완료
  [TEST_08]... 완료
  [TEST_09]... 완료
  [TEST_10]... 완료
  [TEST_11]... 완료
  [TEST_12]... 완료
  [TEST_13]... 완료
  [TEST_14]... 완료
  [TEST_15]... 완료
  [TEST_

# 시도 3

test 시점 역추적이란?

현재 문제

현재 TEST 파일의 시점이 T-8순, T-7순 ... T순 으로 비식별화 되어 있어요.

그래서 모델이 "지금이 몇 월이야?"를 모르는 채로 예측하고 있어요.

SSM 모델의 seasonal=36 은 36순(1년) 주기의 계절성을 학습하는데, 현재 시점을 모르면 "지금이 계절 사이클의 몇 번째 
지점인가"를 제대로 이어붙이지 못해요.

역추적 아이디어
TEST 파일에는 실제 가격값이 들어있어요. 이 가격 패턴을 train의 2022년 이전 데이터와 비교하면 어느 시점인지 추정할 수 있어요.
더 간단한 방법도 있어요. 대회 규칙상 test는 2022년 데이터예요. 그리고 25개 TEST 파일이 순서대로 배치되어 있을 가능성이 높아요.
TEST_00 → 2022년 1월 상순 ~ 3월 하순 (T = 3월 하순)
TEST_01 → 2022년 1월 중순 ~ 4월 상순 (T = 4월 상순)
...
즉 TEST_00의 T순이 언제인지 하나만 찾으면 나머지 24개는 자동으로 계산돼요.

찾는 방법

TEST_00의 가격값과 train의 2022년 전후 시점 가격을 상관관계로 매칭
또는 train의 마지막 시점(2021년 12월 하순) 다음이 2022년 1월 상순임을 이용해 TEST 순서 추정
시점 확정 후 SSM에 정확한 연도·월·순 정보를 넣어서 계절성 예측 정확도 향상

In [3]:
# ============================================================
# 농산물 가격 예측 — 상태 공간 모델 (SSM + 칼만 필터) v5
# v3 대비 수정: TEST 시점 역추적 → 2022년 정확한 위치 파악 후
#               train-gap-test 연속 시계열로 SSM 학습
# ============================================================

import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────────────────────────
# 0. 경로 설정 (★ 여기만 수정하세요 ★)
# ────────────────────────────────────────────────
DATA_DIR = r"C:\Users\LG\DScover\26-1 가이드프로젝트"

def p(filename):
    return os.path.join(DATA_DIR, filename)

def p_meta(filename):
    return os.path.join(DATA_DIR, "meta", filename)


# ────────────────────────────────────────────────
# 1. 공통 데이터 로드
# ────────────────────────────────────────────────
train           = pd.read_csv(p("train.csv"),                   encoding="utf-8-sig")
wholesale_train = pd.read_csv(p("TRAIN_전국도매_2018-2021.csv"), encoding="utf-8-sig")
submission      = pd.read_csv(p("sample_submission.csv"),       encoding="utf-8-sig")

print(f"train: {train.shape} | wholesale: {wholesale_train.shape} | submission: {submission.shape}")


# ────────────────────────────────────────────────
# 2. 타겟 품목 정의
# ────────────────────────────────────────────────
target_items = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '배',          '품종명': '신고',        '거래단위': '10 개',      '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',      '등급': '상품'},
    {'품목명': '무',          '품종명': '무',          '거래단위': '20키로상자', '등급': '상'},
    {'품목명': '상추',        '품종명': '청',          '거래단위': '100 g',      '등급': '상품'},
    {'품목명': '배추',        '품종명': '배추',        '거래단위': '10키로망대', '등급': '상'},
    {'품목명': '양파',        '품종명': '양파',        '거래단위': '1키로',      '등급': '상'},
    {'품목명': '대파',        '품종명': '대파(일반)',  '거래단위': '1키로단',    '등급': '상'},
]

apple_items = [
    {'품목명': '사과', '품종명': '후지', '거래단위': '10 개', '등급': '상품'},
    {'품목명': '사과', '품종명': '홍로', '거래단위': '10 개', '등급': '상품'},
]

wholesale_name_map = {'깐마늘(국산)': '마늘', '건고추': '고추'}

USE_SUPPLY_EXOG = {
    '건고추': False, '사과': True,  '감자': True,
    '배': False,     '깐마늘(국산)': True, '무': False,
    '상추': True,    '배추': False, '양파': False, '대파': True,
}

lag_dict = {
    '건고추': None, '사과': 4,  '감자': 2,
    '배': None,     '깐마늘(국산)': 3, '무': None,
    '상추': 2,      '배추': None, '양파': None, '대파': 2,
}


# ────────────────────────────────────────────────
# 3. 시점 파싱 유틸
# ────────────────────────────────────────────────
def sijeom_to_idx(s):
    """
    '201801상순' → 절대 시점 인덱스 (2018년 1월 상순 = 0)
    2022년 1월 상순 = 144, 2022년 12월 하순 = 179
    """
    decade_map = {'상': 0, '중': 1, '하': 2}
    import re
    m = re.match(r'(\d{4})(\d{2})(상|중|하)순', str(s).strip())
    if not m:
        return None
    year, month, dec = int(m.group(1)), int(m.group(2)), m.group(3)
    return (year - 2018) * 36 + (month - 1) * 3 + decade_map[dec]

def idx_to_sijeom(idx):
    """절대 인덱스 → '202201상순' 형태 문자열"""
    decade_map = {0: '상', 1: '중', 2: '하'}
    year  = 2018 + idx // 36
    month = (idx % 36) // 3 + 1
    dec   = idx % 3
    return f"{year}{month:02d}{decade_map[dec]}순"

# train 시점 → 절대 인덱스 매핑
train_idx_map = {s: sijeom_to_idx(s) for s in train['시점'].unique()}
# train 끝 인덱스: 202112하순 = 143
TRAIN_END_IDX = 143
# 2022년 전체 순 인덱스: 144 ~ 179
YEAR_2022 = list(range(144, 180))


# ────────────────────────────────────────────────
# [수정 v5] TEST 시점 역추적 함수
# ────────────────────────────────────────────────
def estimate_T_idx(test_df, train_df, ref_items):
    """
    test 파일에서 여러 품목의 가격 패턴을 train의 계절 패턴과
    슬라이딩 윈도우 상관관계로 비교해 T순의 절대 인덱스를 추정.

    ref_items: 가격 연속성이 좋은 품목 리스트 (상관 높은 품목 우선)
    반환: T순의 절대 인덱스 (144~179 범위의 2022년 값)
    """
    WINDOW = 9   # T-8순 ~ T순

    scores = {}  # 후보 T_idx → 누적 상관계수

    for item in ref_items:
        cond_tr = (
            (train_df['품목명']   == item['품목명']) &
            (train_df['품종명']   == item['품종명']) &
            (train_df['거래단위'] == item['거래단위']) &
            (train_df['등급']     == item['등급'])
        )
        tr_prices = train_df[cond_tr][['시점', '평균가격(원)']].copy()
        tr_prices['idx'] = tr_prices['시점'].map(train_idx_map)
        tr_prices = tr_prices.dropna(subset=['idx', '평균가격(원)'])
        tr_prices = tr_prices.sort_values('idx')

        # test 가격 추출 (T-8순 ~ T순 순서대로)
        cond_te = (
            (test_df['품목명']   == item['품목명']) &
            (test_df['품종명']   == item['품종명']) &
            (test_df['거래단위'] == item['거래단위']) &
            (test_df['등급']     == item['등급'])
        )
        te_prices = test_df[cond_te]['평균가격(원)'].dropna().values

        if len(te_prices) < 3:
            continue   # 데이터 부족 → 스킵

        te_seq = te_prices[-min(len(te_prices), WINDOW):]

        # train 전체에서 슬라이딩 윈도우로 상관계수 계산
        tr_vals = tr_prices['평균가격(원)'].values
        for start in range(len(tr_vals) - WINDOW + 1):
            window = tr_vals[start: start + WINDOW]
            # 대응하는 train 끝 인덱스 (= 이 window의 T 위치)
            t_candidate = int(tr_prices['idx'].iloc[start + len(window) - 1])

            # 2022년 범위만 후보로 (144~179)
            # 단, train 후반부(2019~2021)도 계절 패턴 참조로 사용
            cmp_len = min(len(te_seq), len(window))
            if cmp_len < 2:
                continue
            try:
                corr = np.corrcoef(te_seq[-cmp_len:], window[-cmp_len:])[0, 1]
                if np.isnan(corr):
                    continue
                # 2022년 해당 계절 위치(같은 월·순)로 매핑
                seasonal_pos = t_candidate % 36   # 0~35: 연내 순 위치
                t_2022 = 144 + seasonal_pos       # 2022년 동일 순 위치
                scores[t_2022] = scores.get(t_2022, 0) + corr
            except Exception:
                continue

    if not scores:
        # 추정 실패 → 중간값(2022년 7월 상순) 반환
        return 162

    best_T = max(scores, key=scores.get)
    return best_T


# 역추적에 쓸 기준 품목 (가격 연속성 높은 품목 우선)
REF_ITEMS_FOR_ESTIMATION = [
    {'품목명': '건고추',      '품종명': '화건',        '거래단위': '30 kg',     '등급': '상품'},
    {'품목명': '깐마늘(국산)','품종명': '깐마늘(국산)','거래단위': '20 kg',      '등급': '상품'},
    {'품목명': '감자',        '품종명': '감자 수미',   '거래단위': '20키로상자', '등급': '상'},
]


# ────────────────────────────────────────────────
# [수정 v5] train + gap(NaN) + test 연속 시계열 구성 함수
# ────────────────────────────────────────────────
def build_continuous_series(df_tr_prices, df_te_prices, T_idx, window=9):
    """
    train 시계열 끝(143)부터 T순(T_idx)까지 갭을 NaN으로 채우고
    test 가격(T-8순~T순)을 올바른 위치에 붙여서 연속 시계열 생성.

    반환: pd.Series (index=절대시점인덱스, values=가격)
    """
    # train 가격 → 절대 인덱스 기반 Series
    tr = df_tr_prices.copy()
    tr['abs_idx'] = tr['시점'].map(train_idx_map)
    tr = tr.dropna(subset=['abs_idx']).set_index('abs_idx')['평균가격(원)']

    # test 가격 → T-8순=T_idx-8 ~ T순=T_idx
    te_vals = df_te_prices['평균가격(원)'].values
    n_te    = len(te_vals)
    te_idx  = list(range(T_idx - n_te + 1, T_idx + 1))
    te      = pd.Series(te_vals, index=te_idx)

    # 전체 인덱스: train 시작(0) ~ T순(T_idx)
    all_idx    = range(0, T_idx + 1)
    full_series = pd.Series(index=all_idx, dtype=float)
    full_series.update(tr)
    full_series.update(te)   # test가 train보다 우선 (최신 데이터)

    return full_series


# ────────────────────────────────────────────────
# 4. 칼만 필터 결측치 대치 (train 전처리) — v3와 동일
# ────────────────────────────────────────────────
def interp_fill(series):
    return series.interpolate(method='linear').ffill().bfill()

def prepare_supply_exog(df_merged, lag_val, steps=3):
    supply      = interp_fill(df_merged['총반입량(kg)'])
    recent_mean = supply.tail(3).mean()
    extended    = pd.concat([supply, pd.Series([recent_mean] * steps)], ignore_index=True)
    shifted     = extended.shift(lag_val).bfill()
    return shifted.iloc[:len(df_merged)].values, shifted.iloc[len(df_merged):].values

print("\n[Step 1] 칼만 필터 결측치 대치 시작...")
all_time_df     = pd.DataFrame({'시점': sorted(train['시점'].unique())})
imputed_results = {}

for item in target_items:
    name     = f"{item['품목명']}_{item['품종명']}"
    col_name = item['품목명']

    cond_price = (
        (train['품목명']   == item['품목명']) &
        (train['품종명']   == item['품종명']) &
        (train['거래단위'] == item['거래단위']) &
        (train['등급']     == item['등급'])
    )
    df_price  = train[cond_price][['시점', '평균가격(원)']]
    df_merged = pd.merge(all_time_df, df_price, on='시점', how='left')

    if USE_SUPPLY_EXOG.get(col_name, False):
        search_name = wholesale_name_map.get(col_name, col_name)
        df_supply   = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_merged   = pd.merge(df_merged, df_supply, on='시점', how='left')
        df_merged['총반입량(kg)'] = interp_fill(df_merged['총반입량(kg)'])

    valid_count = df_merged['평균가격(원)'].notna().sum()
    if valid_count == 0:
        continue

    try:
        model_params = {
            'endog': df_merged['평균가격(원)'],
            'level': 'local linear trend',
            'seasonal': 36, 'autoregressive': 1,
        }
        if USE_SUPPLY_EXOG.get(col_name, False):
            model_params['exog'] = df_merged['총반입량(kg)']

        res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
        df_merged['대치가격(원)'] = df_merged['평균가격(원)'].fillna(res.predict())
        imputed_results[name]     = df_merged
        print(f"  [{name}] 완료 ({valid_count}건 → 144건)")
    except Exception as e:
        print(f"  [{name}] 에러: {e}")

print("✅ 칼만 필터 결측치 대치 완료!\n")


# ────────────────────────────────────────────────
# 5. TEST 추론 (시점 역추적 적용)
# ────────────────────────────────────────────────
print("[Step 2] TEST 추론 시작 (시점 역추적 적용)...")

test_ids   = submission['시점'].apply(lambda x: x.split('+')[0]).unique()
result_sub = submission.copy()
result_sub.set_index('시점', inplace=True)

for test_id in test_ids:
    print(f"  [{test_id}]...", end=" ")
    test_num = test_id.split('_')[1]

    try:
        test           = pd.read_csv(p(f"{test_id}.csv"),                    encoding="utf-8-sig")
        wholesale_test = pd.read_csv(p_meta(f"TEST_전국도매_{test_num}.csv"), encoding="utf-8-sig")
    except FileNotFoundError:
        print("파일 없음 → 건너뜁니다.")
        continue

    # ── [수정 v5] T순 절대 인덱스 추정 ──────────────
    T_idx = estimate_T_idx(test, train, REF_ITEMS_FOR_ESTIMATION)
    T_sijeom = idx_to_sijeom(T_idx)
    print(f"(추정 T시점: {T_sijeom}, idx={T_idx})", end=" ")

    # ── 일반 품목 예측 (사과 제외) ──────────────────
    for item in target_items:
        col_name = item['품목명']
        lag_val  = lag_dict[col_name]

        cond_tr = (
            (train['품목명']   == item['품목명']) &
            (train['품종명']   == item['품종명']) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = (
            (test['품목명']   == item['품목명']) &
            (test['품종명']   == item['품종명']) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        df_te_p = test[cond_te][['시점', '평균가격(원)']].reset_index(drop=True)

        # [수정 v5] train + gap(NaN) + test 연속 시계열 구성
        full_series = build_continuous_series(df_tr_p, df_te_p, T_idx)
        endog       = full_series.values  # NaN 포함 연속 시계열

        # 반입량 exog 구성 (사용 품목만)
        exog_train = exog_future = None
        if USE_SUPPLY_EXOG.get(col_name, False) and lag_val is not None:
            search_name = wholesale_name_map.get(col_name, col_name)
            df_tr_s = (wholesale_train[wholesale_train['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())
            df_te_s = (wholesale_test[wholesale_test['품목명'] == search_name]
                       .groupby('시점')['총반입량(kg)'].sum().reset_index())

            # 반입량도 동일하게 절대 인덱스 기반 연속 시계열 구성
            tr_s = df_tr_s.copy()
            tr_s['abs_idx'] = tr_s['시점'].map(train_idx_map)
            tr_s = tr_s.dropna(subset=['abs_idx']).set_index('abs_idx')['총반입량(kg)']

            te_s_vals = df_te_s['총반입량(kg)'].values
            te_s_idx  = list(range(T_idx - len(te_s_vals) + 1, T_idx + 1))
            te_s      = pd.Series(te_s_vals, index=te_s_idx)

            supply_series = pd.Series(index=range(len(endog)), dtype=float)
            supply_series.update(tr_s)
            supply_series.update(te_s)
            supply_series = interp_fill(supply_series)

            # 시차 적용
            recent_mean  = supply_series.tail(3).mean()
            extended     = pd.concat(
                [supply_series, pd.Series([recent_mean] * 3)], ignore_index=True
            )
            shifted      = extended.shift(lag_val).bfill()
            exog_train   = shifted.iloc[:len(endog)].values
            exog_future  = shifted.iloc[len(endog):].values.reshape(-1, 1) \
                           if len(shifted) > len(endog) else None

        try:
            model_params = {
                'endog':          endog,
                'level':          'local linear trend',
                'seasonal':       36,
                'autoregressive': 1,
            }
            if exog_train is not None:
                model_params['exog'] = exog_train

            res = sm.tsa.UnobservedComponents(**model_params).fit(disp=False)
            fc  = np.maximum(
                res.forecast(steps=3, exog=exog_future).values
                if exog_future is not None
                else res.forecast(steps=3).values,
                0
            )
            result_sub.loc[f'{test_id}+1순', col_name] = fc[0]
            result_sub.loc[f'{test_id}+2순', col_name] = fc[1]
            result_sub.loc[f'{test_id}+3순', col_name] = fc[2]

        except Exception:
            result_sub.loc[f'{test_id}+1순', col_name] = 0
            result_sub.loc[f'{test_id}+2순', col_name] = 0
            result_sub.loc[f'{test_id}+3순', col_name] = 0

    # ── 사과 홍로/후지 분리 예측 ──
    apple_forecasts = {}
    for item in apple_items:
        variety = item['품종명']
        cond_tr = (
            (train['품목명']   == '사과') &
            (train['품종명']   == variety) &
            (train['거래단위'] == item['거래단위']) &
            (train['등급']     == item['등급'])
        )
        df_tr_p = train[cond_tr][['시점', '평균가격(원)']]

        cond_te = (
            (test['품목명']   == '사과') &
            (test['품종명']   == variety) &
            (test['거래단위'] == item['거래단위']) &
            (test['등급']     == item['등급'])
        )
        df_te_p = test[cond_te][['시점', '평균가격(원)']].reset_index(drop=True)

        full_series = build_continuous_series(df_tr_p, df_te_p, T_idx)
        endog       = full_series.values

        df_tr_s = (wholesale_train[
                        (wholesale_train['품목명'] == '사과') &
                        (wholesale_train['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())
        df_te_s = (wholesale_test[
                        (wholesale_test['품목명'] == '사과') &
                        (wholesale_test['품종명'] == variety)
                   ].groupby('시점')['총반입량(kg)'].sum().reset_index())

        tr_s = df_tr_s.copy()
        tr_s['abs_idx'] = tr_s['시점'].map(train_idx_map)
        tr_s = tr_s.dropna(subset=['abs_idx']).set_index('abs_idx')['총반입량(kg)']
        te_s_vals = df_te_s['총반입량(kg)'].values
        te_s_idx  = list(range(T_idx - len(te_s_vals) + 1, T_idx + 1))
        te_s      = pd.Series(te_s_vals, index=te_s_idx)
        supply_series = pd.Series(index=range(len(endog)), dtype=float)
        supply_series.update(tr_s)
        supply_series.update(te_s)
        supply_series = interp_fill(supply_series)

        recent_mean = supply_series.tail(3).mean()
        extended    = pd.concat([supply_series, pd.Series([recent_mean] * 3)], ignore_index=True)
        shifted     = extended.shift(lag_dict['사과']).bfill()
        exog_train  = shifted.iloc[:len(endog)].values
        exog_future = shifted.iloc[len(endog):].values if len(shifted) > len(endog) else None

        try:
            res = sm.tsa.UnobservedComponents(
                endog=endog, level='local linear trend',
                seasonal=36, autoregressive=1, exog=exog_train,
            ).fit(disp=False)
            fc = np.maximum(
                res.forecast(steps=3, exog=exog_future).values
                if exog_future is not None
                else res.forecast(steps=3).values,
                0
            )
            apple_forecasts[variety] = fc
        except Exception:
            apple_forecasts[variety] = np.array([0.0, 0.0, 0.0])

    fc_apple = np.mean(list(apple_forecasts.values()), axis=0) if apple_forecasts else np.zeros(3)
    result_sub.loc[f'{test_id}+1순', '사과'] = fc_apple[0]
    result_sub.loc[f'{test_id}+2순', '사과'] = fc_apple[1]
    result_sub.loc[f'{test_id}+3순', '사과'] = fc_apple[2]

    print("완료")


# ────────────────────────────────────────────────
# 6. 저장
# ────────────────────────────────────────────────
result_sub.reset_index(inplace=True)
out_path = p("submission_SSM_v5_time_estimation.csv")
result_sub.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n저장 완료 → {out_path}")
print(f"0인 값 개수: {(result_sub.iloc[:,1:] == 0).sum().sum()}")
print(result_sub.head(6).to_string())


# ============================================================
# ## 수정 내역 (v3 → v5)
#
# ### 핵심 문제 (v3까지의 한계)
# - test 시점이 'T-8순 ~ T순'으로 비식별화되어 있어
#   train 끝(2021년 12월 하순)과 test 사이의 갭을 모름
# - 기존 코드는 `pd.concat([train, test])`로 그냥 이어붙여서
#   SSM이 2021년 12월 하순 바로 다음이 test인 것으로 잘못 인식
# - → 계절성(seasonal=36) 예측의 시작점이 틀려서 오차 발생
#
# ### 수정 1: TEST 시점 역추적 (estimate_T_idx)
# - 건고추·깐마늘·감자 등 가격 연속성 높은 품목의 9순 가격 패턴을
#   train 전체에 슬라이딩 윈도우로 비교 → 상관계수 최대 위치 탐색
# - 해당 위치의 계절 위치(월·순)를 2022년으로 변환 → T순 추정
#
# ### 수정 2: train + gap(NaN) + test 연속 시계열 구성 (build_continuous_series)
# - 기존: train(144) + test(9) 단순 concat (갭 무시)
# - 변경: train(144) + 갭(NaN) + test(9)를 절대 인덱스 기반으로 올바르게 배치
# - 갭 구간(2021년 12월 하순 ~ T-8순 사이)은 NaN → 칼만 필터가 자동 추론
# - SSM이 "지금이 계절 사이클의 몇 번째인가"를 정확하게 인식
#
# ### 파일명
# - submission_SSM_v5_time_estimation.csv
# ============================================================

train: (29376, 7) | wholesale: (176014, 22) | submission: (75, 11)

[Step 1] 칼만 필터 결측치 대치 시작...
  [건고추_화건] 완료 (144건 → 144건)
  [감자_감자 수미] 완료 (144건 → 144건)
  [배_신고] 완료 (144건 → 144건)
  [깐마늘(국산)_깐마늘(국산)] 완료 (144건 → 144건)
  [무_무] 완료 (144건 → 144건)
  [상추_청] 완료 (144건 → 144건)
  [배추_배추] 완료 (144건 → 144건)
  [양파_양파] 완료 (144건 → 144건)
  [대파_대파(일반)] 완료 (144건 → 144건)
✅ 칼만 필터 결측치 대치 완료!

[Step 2] TEST 추론 시작 (시점 역추적 적용)...
  [TEST_00]... (추정 T시점: 202210상순, idx=171) 완료
  [TEST_01]... (추정 T시점: 202207중순, idx=163) 완료
  [TEST_02]... (추정 T시점: 202211하순, idx=176) 완료
  [TEST_03]... (추정 T시점: 202205하순, idx=158) 완료
  [TEST_04]... (추정 T시점: 202203하순, idx=152) 완료
  [TEST_05]... (추정 T시점: 202206중순, idx=160) 완료
  [TEST_06]... (추정 T시점: 202204상순, idx=153) 완료
  [TEST_07]... (추정 T시점: 202210상순, idx=171) 완료
  [TEST_08]... (추정 T시점: 202206중순, idx=160) 완료
  [TEST_09]... (추정 T시점: 202211하순, idx=176) 완료
  [TEST_10]... (추정 T시점: 202203하순, idx=152) 완료
  [TEST_11]... (추정 T시점: 202205하순, idx=158) 완료
  [TEST_12]... (추정 T시점: 202206상순, idx=15